# Static MB calibration against WGMS data with AdaPBS

OGGM's `mb_calibration_to_rmsd` finds the mass balance (MB) parameters
(`melt_f`, `prcp_fac`, `temp_bias`) which minimise the RMSD between the modelled
and the observed (WGMS) annual MB of a glacier. It returns one parameter set.

Here we do the Bayesian version of this with the adaptive particle batch smoother
(AdaPBS): given a prior distribution of the parameters and observations with
errors, it returns weighted samples of the **posterior** distribution. This tells
us not only the best parameters, but also how well the observations constrain
them, and which combinations of parameters are equally good (equifinality).

The geometry is fixed (the inversion flowlines), as in `mb_calibration_to_rmsd`:
one model run is a few milliseconds, so we can afford thousands of them.

The notebook has two parts:

1. **One target: the annual MB series.** The AdaPBS equivalent of
   `mb_calibration_to_rmsd`. We use it to explain how AdaPBS works, and check
   that it finds the same solution as the deterministic calibration.
2. **Several targets: the winter and summer MB.** Where the Bayesian approach
   becomes useful: we combine observations of different kinds, and see what each
   of them constrains.

AdaPBS is the work of K. Aalstad and E. Alonso-González (MuSA). If you use it,
please cite Aalstad et al. (2026, https://doi.org/10.5194/gmd-19-8565-2026) and
Alonso-González et al. (2022, https://doi.org/10.5194/gmd-15-9127-2022).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from oggm import cfg, utils, workflow
from oggm.core.massbalance import mb_calibration_to_rmsd

from oggm_adapbs import run_adapbs, glogit, gexpit, weighted_quantile, weighted_summary
from oggm_adapbs.massbalance import mb_prior, static_mb_model, seasonal_specific_mb

## Glacier and data

We use Vernagtferner (Ötztal Alps, Austria), which has 40 years of winter and
summer MB in OGGM's WGMS data over the period of the W5E5 climate data
(1979-2019). Its neighbour Hintereisferner (`RGI60-11.00897`) works too, but has
seasonal data since 2013 only.

We start from the OGGM preprocessed directories at level 3 (W5E5 climate,
elevation band flowlines).

In [ ]:
cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = utils.gettempdir('adapbs_wgms_calibration', reset=True)

rgi_id = 'RGI60-11.00719'  # Vernagtferner
base_url = ('https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/'
            'L3-L5_files/2025.6/elev_bands/W5E5/per_glacier/')
gdir = workflow.init_glacier_directories([rgi_id], from_prepro_level=3,
                                         prepro_border=80,
                                         prepro_base_url=base_url)[0]

# The temperature bias file used by OGGM to inform the temp_bias prior
temp_bias_file = ('https://cluster.klima.uni-bremen.de/~oggm/ref_mb_params/oggm_v1.6/'
                  'w5e5_rgi6_perglacier_temp_bias_v2025.6.2.csv')
gdir.name

The WGMS MB is given for hydrological years (October to September), with a
winter (October to April) and a summer (May to September) balance. We keep the
years with all three values. The climate data end in 2019.

In [ ]:
ref = gdir.get_ref_mb_data(y0=1980, y1=2019)
ref = ref[['WINTER_BALANCE', 'SUMMER_BALANCE', 'ANNUAL_BALANCE']].dropna()
ref.columns = ['winter', 'summer', 'annual']
years = ref.index.values
print(f'{len(years)} years: {years[0]}-{years[-1]}')
ref.describe().round(0)

# Part 1: one target, the annual MB series

## Deterministic reference: `mb_calibration_to_rmsd`

We calibrate the three parameters together, with OGGM's default bounds. Note
that `mb_calibration_to_rmsd` compares the WGMS data (hydrological years) to the
model's calendar years, while we use hydrological years for AdaPBS below.

In [ ]:
params = ['melt_f', 'prcp_fac', 'temp_bias']
det = mb_calibration_to_rmsd(gdir, ref_df=ref['annual'], calibrate_params=tuple(params),
                             write_to_gdir=False, optimisation_kwargs={'seed': 0})
print({p: round(det[p], 2) for p in params}, f"RMSD: {det['rmsd']:.0f} kg m-2")

## The prior

### Why do we need a prior?

In Bayesian inference, the posterior distribution of the parameters is
proportional to the **prior** times the **likelihood**:

$$p(\theta \mid \mathrm{obs}) \propto p(\theta) \; p(\mathrm{obs} \mid \theta)$$

The likelihood says how well a parameter set $\theta$ = (`melt_f`, `prcp_fac`,
`temp_bias`) explains the observations. The prior says which parameter values we
find plausible *before* looking at the observations: for example, that `prcp_fac`
is probably close to the value estimated from the winter precipitation, and
certainly not 0.01 or 50.

We need it for two reasons:

- **The observations alone are often not enough.** When several parameter sets
  fit the observations equally well, the prior decides between them. The
  deterministic calibration has a prior too, in disguise: the bounds of the
  optimisation, and the values at which the parameters which are not calibrated
  are fixed.
- **AdaPBS needs somewhere to start**: it draws its first batch of particles
  (parameter sets) from the prior.

The prior does matter for the result. We will see an example of this at the
end of this notebook (section *How much does the prior matter?*).

### The prior in practice

Each parameter has a **logit-normal** prior. This is a Gaussian distribution
in a transformed space (the "normal space"), which becomes a bounded distribution
in the parameter space: values outside the bounds are impossible. The transform
is a generalised logit function, `glogit`, and its inverse is `gexpit`.

`mb_prior` (from `oggm_adapbs.massbalance`) defines the centre and bounds of the
prior of each parameter. The centres are OGGM's informed guesses:

- `melt_f`: 5 (`cfg.PARAMS['melt_f']`)
- `prcp_fac`: from the winter precipitation (as in OGGM's informed three-step
  calibration)
- `temp_bias`: from the temperature bias file

For the comparison with `mb_calibration_to_rmsd`, we give the prior the same
bounds as the deterministic calibration (OGGM's default bounds, from the glacier
settings), with the `prior_bounds` argument. `mb_prior` has narrower default
bounds, which we will use at the end of the notebook.

The width of the prior is given by its standard deviation in the normal space:
we use 1 for all parameters (more on this below).

In [ ]:
# OGGM's default bounds, as used by mb_calibration_to_rmsd
oggm_bounds = {p: (gdir.settings[f'{p}_min'], gdir.settings[f'{p}_max']) for p in params}

# The centre and bounds of the prior of each parameter
prior = mb_prior(gdir, params, temp_bias_file_path=temp_bias_file,
                 prior_bounds=oggm_bounds)
x_min, x_max = prior['x_min'], prior['x_max']
prior

`glogit` transforms values from the parameter space to the normal space, and
`gexpit` does the opposite. The middle of the bounds is 0 in the normal space,
the bounds themselves are at $-\infty$ and $+\infty$:

In [ ]:
# The prior centre, the middle of the bounds, and values close to the bounds
x = np.array([prior['x_centre'], (x_min + x_max) / 2, x_min + 0.01, x_max - 0.01])
z = glogit(x, x_min, x_max)
pd.DataFrame(z, columns=params, index=['centre', 'middle', 'near min', 'near max']).round(2)

The prior is a Gaussian centred on the normal-space value of the prior centre,
with a standard deviation of 1. To see what this means for the parameters, we
draw many samples of it in the normal space and transform them back with
`gexpit`.

`weighted_summary` computes quantiles and the mean of each parameter from samples
and their weights: here all samples have the same weight. We will use it again
for the posterior, where the weights differ.

In [ ]:
prior_mean = glogit(prior['x_centre'], x_min, x_max)  # the centre, in normal space
prior_sd = 1.

# Draw 10000 samples in normal space, and transform them to parameter space
rng = np.random.default_rng(0)
prior_theta = gexpit(prior_mean + prior_sd * rng.standard_normal((10000, 3)), x_min, x_max)

# Quantiles of the prior, with equal weights
prior_df = weighted_summary(prior_theta, np.ones(10000) / 10000, params)
prior_df['centre'], prior_df['min'], prior_df['max'] = prior['x_centre'], x_min, x_max
prior_df.round(2)

The median of the prior is its centre, and 90% of the prior lies between the
`q5` and `q95` values. With these wide bounds, the prior is not very informative:
it only says "probably not too far from the centre".

A standard deviation of 1 in normal space gives a broad, single-peaked
distribution. Above ~1.8, the logit-normal distribution puts most of its mass
close to the bounds, which is rarely what we want.

## The observation error

The likelihood compares the predicted and the observed values, and needs to know
how close we expect them to be: the **observation error**. We assume
independent Gaussian errors. The error must include the measurement error *and*
the model error (the MB model with gridded climate data cannot reproduce every
year): the latter dominates.

We use the RMSD of the deterministic calibration as the error of each annual
value, $\sigma_a$. With this choice, the most likely parameters of AdaPBS are
those which minimise the RMSD: the two calibrations should agree.

This is a choice: the width of the posterior scales with it. Also, the model
errors of consecutive years are not independent in reality, which makes the
posterior too narrow.

In [ ]:
n = len(years)
sigma_a = det['rmsd']
f'sigma_a = {sigma_a:.0f} kg m-2'

For part 2, we define the observations and errors of each experiment with two
small helper functions: `observe` picks the observed quantities out of a table of
winter, summer and annual MB (we use it for both the observations and the model
output), and `obs_errors` gives their errors. The seasonal errors are explained
in part 2.

In [ ]:
sigma = {'annual': sigma_a, 'winter': sigma_a / np.sqrt(2), 'summer': sigma_a / np.sqrt(2)}

def observe(df, target):
    # The quantities used by an experiment, from a DataFrame with the columns
    # winter, summer, annual (one row per year). E.g. 'winter+summer': the
    # winter series, followed by the summer series
    return np.concatenate([df[k].values for k in target.split('+')])

def obs_errors(target):
    # The errors (1 sigma) of these quantities
    return np.concatenate([np.repeat(sigma[k], n) for k in target.split('+')])

# For example: 39 annual values, or 39 + 39 seasonal values
print(len(observe(ref, 'annual')), len(observe(ref, 'winter+summer')))

## The forward model

AdaPBS does not know anything about glaciers: it only needs a **forward
model**, a function which takes a batch of parameter sets and returns the
predicted observations for each of them. Here, the forward model is OGGM's MB
model on a fixed geometry.

`static_mb_model` creates the MB model (here `MonthlyTIModel`) of the glacier and
reads its inversion flowlines. We start it with the prior centre and change
the parameters later.

In [ ]:
mb_mod, fls = static_mb_model(gdir, prior['centre'])
mb_mod.melt_f, mb_mod.prcp_fac, mb_mod.temp_bias

OGGM's `get_specific_mb` gives the MB of calendar years, but the WGMS data are
given for hydrological years, with seasonal values. `seasonal_specific_mb` sums
the monthly MB of the model into winter (October to April), summer (May to
September) and annual (October to September) values, in kg m-2:

In [ ]:
# The MB of the glacier with the parameters at the prior centre
seasonal_specific_mb(mb_mod, fls, years).head()

Now the forward function itself. AdaPBS calls it as `forward(theta, iteration)`:

- `theta` is an array with one row per particle and one column per parameter
  (in the order of `params`), in the parameter space;
- `iteration` is the number of the AdaPBS iteration (starting at 0). Our MB model
  does not need it, but some forward models do: for example, a dynamical model
  run which writes output files needs unique file names for each particle of
  each iteration. It can also be used to print progress;
- it must return the predicted observations, one row per particle (NaN for a
  failed model run: such a particle gets a zero weight).

It can also return a dictionary of "extras", which AdaPBS stores with the
results: we use it to keep the full winter, summer and annual series of each
particle, to look at the fit afterwards.

We create one forward function per experiment with `make_forward`.

In [ ]:
seasons = ['winter', 'summer', 'annual']

def make_forward(target):
    def forward(theta, iteration):
        pred, extras = [], {k: [] for k in seasons}
        for th in theta:  # one particle (parameter set) at a time
            # Set the parameters of the MB model
            for p, v in zip(params, th):
                setattr(mb_mod, p, v)
            # Run the model and keep what we need
            df = seasonal_specific_mb(mb_mod, fls, years)
            for k in seasons:
                extras[k].append(df[k].values)
            pred.append(observe(df, target))
        return np.array(pred), {k: np.array(v) for k, v in extras.items()}
    return forward

# Test it with three particles: the mean of the predicted annual MB of each
forward = make_forward('annual')
pred, extras = forward(np.array([[5, 3, 1.5], [5, 3, 2.5], [5, 4, 1.5]]), 0)
pred.shape, pred.mean(axis=1)

39 predicted values (years) for each of the 3 particles. Warmer (higher
`temp_bias`) means more negative MB, more precipitation (higher `prcp_fac`)
means more positive MB: the model responds as expected.

## Running AdaPBS

AdaPBS works in iterations:

1. draw a batch of particles (300 here) from a proposal distribution: the prior
   at the first iteration;
2. run the forward model for each particle;
3. compute the weight of *all* the particles so far, from all iterations: how
   likely they are under the posterior (prior times likelihood), relative to how
   likely they were to be drawn (importance sampling);
4. if the weights are concentrated on a few particles, build a new proposal
   (a Gaussian in normal space) from the best particles, and go back to 1.
   Otherwise, stop.

The result is the set of all particles and their weights: a weighted sample of
the posterior.

How concentrated the weights are is measured by the **effective sample size**,
$N_{eff} = 1 / \sum w_i^2$: it is the number of particles if all have the same
weight, and 1 if one particle has all the weight. AdaPBS stops when $N_{eff}$ is
above a fraction (here 30%) of the number of particles per iteration.

`run_adapbs` does all this. Its main arguments are the forward function, the
observations and their errors, the bounds of the parameters and the prior
(centre and standard deviation, in normal space). We fix the random seed for
reproducible results.

In [ ]:
res = run_adapbs(
    make_forward('annual'),         # the forward model
    observe(ref, 'annual'),         # the observations
    obs_errors('annual'),           # their errors (1 sigma)
    x_min, x_max,                   # the parameter bounds
    prior_mean=prior_mean,          # the prior centre (normal space)
    prior_sd=prior_sd,              # the prior standard deviation (normal space)
    n_particles=300,                # particles per iteration
    max_iterations=10,              # stop after 10 iterations at most
    neff_threshold=0.3,             # stop when Neff > 30% of n_particles
    rng=np.random.default_rng(0),   # random generator (reproducible)
)

`run_adapbs` returns a dictionary. Its main entries are:

- `theta`: all the particles of all iterations (one row each, parameter space)
- `weights`: their posterior weights (they sum to 1)
- `iteration`: the iteration of each particle (starting at 1)
- `pred`: their predicted observations
- `extras`: what our forward function returned as extras
- `posterior_theta`: 300 particles resampled according to the weights (see
  below)
- `diagnostics`: a table with one row per iteration

In [ ]:
res['diagnostics'][['iteration', 'n_history', 'neff', 'neff_frac']]

At the first iteration (particles drawn from the prior), Neff = 1: a single
particle carries almost all the weight. The observations (39 values with an
error of 250 kg m-2 each) are much more precise than the prior, so very few
samples of the prior fall where the observations allow. AdaPBS then builds new
proposals from the best particles: they move towards the posterior, and Neff
grows until it passes 30% of 300 (90), at the 6th iteration. In total: 1800
model runs.

The figure shows this in the `melt_f` - `prcp_fac` plane:

In [ ]:
n_it = len(res['diagnostics'])
fig, axes = plt.subplots(1, n_it, figsize=(3 * n_it, 3.2), sharex=True, sharey=True)
for j, ax in enumerate(axes):
    it = res['iteration']
    ax.scatter(prior_theta[:1000, 0], prior_theta[:1000, 1], s=3, color='0.85')
    # Particles of the previous iterations
    old = it < j + 1
    ax.scatter(res['theta'][old, 0], res['theta'][old, 1], s=4, color='0.55')
    # Particles drawn at this iteration
    new = it == j + 1
    ax.scatter(res['theta'][new, 0], res['theta'][new, 1], s=6, color='#2a78d6')
    ax.scatter(det['melt_f'], det['prcp_fac'], marker='X', s=80, color='k',
               edgecolor='w', zorder=5)
    neff = res['diagnostics']['neff'].iloc[j]
    ax.set_title(f'iteration {j + 1}\nNeff = {neff:.0f}', fontsize=10)
    ax.set_xlabel('melt_f')
axes[0].set_ylabel('prcp_fac')
axes[0].set_xlim(1.5, 12)
axes[0].set_ylim(0, 6)
plt.tight_layout()

Light grey: samples of the prior. Blue: the particles drawn at each iteration.
Dark grey: the particles of the previous iterations. Black cross:
`mb_calibration_to_rmsd`.

### Weights and resampled particles

The weights of the last iteration are the posterior weights of all the
particles, from all iterations. How many of them really count?

In [ ]:
w = np.sort(res['weights'])[::-1]
n90 = np.searchsorted(np.cumsum(w), 0.9) + 1
print(f'{len(w)} particles, {n90} of them carry 90% of the weight')

Most particles have a weight close to zero: the first iterations were mostly
exploring the prior.

Weighted particles are not always convenient (for example, to run a glacier
model with each of them). `posterior_theta` contains 300 particles resampled
with replacement according to the weights: each of them has the same weight,
and together they are a sample of the posterior. Particles with a large weight
can appear several times.

## The posterior

We summarise the posterior of each parameter by its median and 5-95% range,
computed from the weighted particles with `weighted_summary`:

In [ ]:
def fmt(row):
    return f"{row['q50']:.2f} [{row['q5']:.2f}, {row['q95']:.2f}]"

table = pd.DataFrame({'prior': prior_df.apply(fmt, axis=1),
                      'AdaPBS annual': weighted_summary(res['theta'], res['weights'],
                                                        params).apply(fmt, axis=1),
                      'mb_calibration_to_rmsd': [f'{det[p]:.2f}' for p in params]}).T
table

In [ ]:
labels = {'melt_f': 'melt_f (kg m-2 day-1 K-1)', 'prcp_fac': 'prcp_fac',
          'temp_bias': 'temp_bias (K)'}
plot_range = {'melt_f': (1.5, 12), 'prcp_fac': (0, 6), 'temp_bias': (-4, 7)}

def plot_marginals(results, colors):
    # Weighted histograms of the particles, one panel per parameter
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    for k, (ax, p) in enumerate(zip(axes, params)):
        bins = np.linspace(*plot_range[p], 60)
        ax.hist(prior_theta[:, k], bins=bins, density=True, color='0.8', label='prior')
        for t, r in results.items():
            ax.hist(r['theta'][:, k], bins=bins, weights=r['weights'], density=True,
                    histtype='step', lw=2, color=colors[t], label=t)
        ax.axvline(det[p], color='k', ls='--', lw=1.5, label='mb_calibration_to_rmsd')
        ax.set_xlabel(labels[p])
        ax.set_yticks([])
    axes[0].set_ylabel('density')
    axes[-1].legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
    plt.tight_layout()

def plot_pairs(results, colors):
    # The resampled particles, for each pair of parameters
    pairs = [('melt_f', 'prcp_fac'), ('melt_f', 'temp_bias'), ('prcp_fac', 'temp_bias')]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, (px, py) in zip(axes, pairs):
        ix, iy = params.index(px), params.index(py)
        ax.scatter(prior_theta[:2000, ix], prior_theta[:2000, iy], s=4, color='0.85',
                   label='prior')
        for t, r in results.items():
            post = r['posterior_theta']
            ax.scatter(post[:, ix], post[:, iy], s=8, color=colors[t], alpha=0.6,
                       label=t)
        ax.scatter(det[px], det[py], marker='X', s=120, color='k',
                   edgecolor='w', label='mb_calibration_to_rmsd', zorder=5)
        ax.set_xlim(plot_range[px])
        ax.set_ylim(plot_range[py])
        ax.set_xlabel(labels[px])
        ax.set_ylabel(labels[py])
    axes[-1].legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False,
                    markerscale=2)
    plt.tight_layout()

colors = {'annual': '#eb6834', 'winter': '#1baf7a', 'summer': '#eda100',
          'winter+summer': '#e87ba4'}
results = {'annual': res}
plot_marginals(results, colors)
plot_pairs(results, colors)

AdaPBS finds the same solution as `mb_calibration_to_rmsd`: the deterministic
parameters are close to the posterior median. In addition, it tells us how well
the annual MB constrains the parameters: the posterior is much narrower than the
prior. The pair plots show that the parameters are correlated: a higher
`melt_f` goes with a lower `temp_bias` (more melt per degree, fewer degrees),
and a higher `prcp_fac` with a higher `temp_bias` (more snow, compensated by
more melt).

## Fit to the observations

How well does the model fit the observations with the posterior parameters?
The extras give us the modelled series of each particle. For each year, we
compute the 5-95% range of the modelled MB over the weighted particles with
`weighted_quantile` (the posterior predictive distribution), and compare it to
the WGMS data. We also look at the winter and summer MB, which we did not use
for the calibration.

In [ ]:
def band(r, season, q):
    # Weighted quantile q of the modelled MB, for each year
    x = r['extras'][season]  # one row per particle, one column per year
    return np.array([weighted_quantile(x[:, i], r['weights'], q)
                     for i in range(x.shape[1])])

# The MB with the parameters of the deterministic calibration
for p in params:
    setattr(mb_mod, p, det[p])
det_mb = seasonal_specific_mb(mb_mod, fls, years)

def plot_fit(results, colors):
    fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
    for ax, s in zip(axes, seasons):
        for t, r in results.items():
            ax.fill_between(years, band(r, s, 0.05), band(r, s, 0.95),
                            color=colors[t], alpha=0.35, lw=0, label=f'AdaPBS {t} (5-95%)')
        ax.plot(years, det_mb[s], color='k', ls='--', lw=1.5, label='mb_calibration_to_rmsd')
        ax.plot(years, ref[s], 'o', color='k', ms=5, label='WGMS')
        ax.set_ylabel(f'{s} MB (kg m-2)')
    axes[0].legend(ncol=2, loc='upper left', frameon=False, fontsize=9)
    axes[0].set_ylim(top=axes[0].get_ylim()[1] + 600)
    plt.tight_layout()

def fit_table(results):
    # RMSD and mean bias of the posterior mean MB, per season
    rows = {}
    for t, r in results.items():
        for s in seasons:
            m = np.average(r['extras'][s], axis=0, weights=r['weights'])
            rows[(t, s)] = {'RMSD': np.sqrt(np.mean((m - ref[s]) ** 2)),
                            'bias': np.mean(m - ref[s])}
    for s in seasons:
        rows[('mb_calibration_to_rmsd', s)] = {
            'RMSD': np.sqrt(np.mean((det_mb[s] - ref[s]) ** 2)),
            'bias': np.mean(det_mb[s] - ref[s])}
    return pd.DataFrame(rows).T.unstack().round(0)

plot_fit(results, colors)
fit_table(results)

Both calibrations fit the annual MB equally well (RMSD 250 kg m-2). But they
get the seasons wrong: the winter MB is too low by ~530 kg m-2, and the summer
MB too high by the same amount. The model reaches the right annual MB with too
little accumulation and too little melt. The annual MB cannot see this: it only
constrains the difference between accumulation and melt. This is where part 2
comes in.

# Part 2: several targets, the winter and summer MB

The winter MB depends mostly on the accumulation (`prcp_fac`), the summer MB
mostly on the melt (`melt_f` and `temp_bias`). Using them as observations should
help where the annual MB cannot. We run three experiments:

| experiment | observations |
|---|---|
| `winter` | the winter MB time series |
| `summer` | the summer MB time series |
| `winter+summer` | both seasonal series together |

Errors: $\sigma_a / \sqrt{2}$ for each seasonal value, so that the two seasons
together carry the same error as the annual MB (if the two errors are
independent). The prior is the same as in part 1.

In [ ]:
{k: round(v) for k, v in sigma.items()}

In [ ]:
for target in ['winter', 'summer', 'winter+summer']:
    results[target] = run_adapbs(make_forward(target), observe(ref, target),
                                 obs_errors(target), x_min, x_max,
                                 prior_mean=prior_mean, prior_sd=prior_sd,
                                 n_particles=300, max_iterations=10,
                                 neff_threshold=0.3, rng=np.random.default_rng(0))

pd.DataFrame({t: {'iterations': len(r['diagnostics']),
                  'model runs': len(r['weights']),
                  'final Neff': r['diagnostics']['neff'].iloc[-1]}
              for t, r in results.items()}).T.round(0)

In [ ]:
table = {'prior': prior_df.apply(fmt, axis=1)}
for t, r in results.items():
    table[t] = weighted_summary(r['theta'], r['weights'], params).apply(fmt, axis=1)
table = pd.DataFrame(table).T
table.loc['mb_calibration_to_rmsd'] = [f'{det[p]:.2f}' for p in params]
table

In [ ]:
plot_marginals(results, colors)

In [ ]:
plot_pairs({t: results[t] for t in ['annual', 'winter+summer']}, colors)

In [ ]:
plot_fit({t: results[t] for t in ['annual', 'winter+summer']}, colors)
fit_table({t: results[t] for t in ['annual', 'winter+summer']})

What does each experiment constrain?

- **`winter`** constrains `prcp_fac` (2.56 [2.41, 2.67]), but not `melt_f`: its
  posterior is the prior. `temp_bias` is only loosely constrained (it changes
  the fraction of solid precipitation): its posterior extends beyond the plot,
  down to -10 K.
- **`summer`** constrains `melt_f` and `temp_bias`, and also prefers a low
  `prcp_fac` (summer snowfall is part of the summer balance).
- **`winter+summer`** constrains all three parameters: `prcp_fac` is about twice
  the value of the `annual` experiment. The pair plots show two clearly separate
  solutions. `melt_f` and `temp_bias` still trade off along a ridge, but a
  narrower one.

The fit: the winter and summer biases drop from +-530 to less than 60 kg m-2.
The annual RMSD increases from 250 to 316 kg m-2: this is the price for a model
which gets both seasons right.

# How much does the prior matter?

So far, the prior had wide bounds, the same as the deterministic calibration.
`mb_prior` has narrower default bounds, closer to what we consider physically
plausible: `prcp_fac` between half and twice its centre, `temp_bias` +- 3 K
around its centre (`melt_f` keeps OGGM's bounds). Let's run the `annual` and
`winter+summer` experiments again with this prior.

In [ ]:
narrow = mb_prior(gdir, params, temp_bias_file_path=temp_bias_file)
narrow_mean = glogit(narrow['x_centre'], narrow['x_min'], narrow['x_max'])
print('prcp_fac bounds:', narrow['x_min'][1].round(2), narrow['x_max'][1].round(2))
print('temp_bias bounds:', narrow['x_min'][2].round(2), narrow['x_max'][2].round(2))

results_narrow = {}
for target in ['annual', 'winter+summer']:
    results_narrow[target] = run_adapbs(make_forward(target), observe(ref, target),
                                        obs_errors(target), narrow['x_min'], narrow['x_max'],
                                        prior_mean=narrow_mean, prior_sd=prior_sd,
                                        n_particles=300, max_iterations=10,
                                        neff_threshold=0.3, rng=np.random.default_rng(0))

table = {}
for t in ['annual', 'winter+summer']:
    for name, rr in [('wide prior', results), ('narrow prior', results_narrow)]:
        r = rr[t]
        table[(t, name)] = weighted_summary(r['theta'], r['weights'], params).apply(fmt, axis=1)
pd.DataFrame(table).T

With the narrow prior, the `annual` posterior of `prcp_fac` piles up against
its lower bound (1.68): median 1.96, against 1.13 with the wide prior. And
`temp_bias` moves to compensate (2.49 against 1.55). The annual MB wants a low
`prcp_fac`, the prior does not allow it: the result is a compromise decided by
the prior.

The `winter+summer` posterior does not change: the winter MB constrains
`prcp_fac` well inside both priors.

This is general: the prior matters most for the parameters (or combinations of
parameters) which the observations do not constrain well. When a posterior
piles up against a prior bound, the prior is doing part of the job: ask whether
the bound is physically justified, or whether you need more observations.

A more systematic look at the role of the prior (its centre, width and bounds)
will be the topic of another notebook.

# Take-aways

- With one target (the annual MB series), AdaPBS reproduces
  `mb_calibration_to_rmsd`: the same parameters, within the posterior range, and
  the same RMSD. In addition, it gives the uncertainty of the parameters and
  their correlations.
- It needed 1800 model runs (6 iterations) for this: about 20 seconds on a
  laptop with the static MB model.
- The annual MB alone gives the right annual MB with too small a seasonal
  amplitude (too little accumulation and melt, seasonal biases of +-530 kg m-2),
  and cannot tell.
- The winter and summer MB together constrain all three parameters, and the
  model fits both seasons. Combining observations of different kinds is where
  the Bayesian approach is most useful.
- The prior matters where the observations are not informative enough: here,
  for `prcp_fac` with the annual MB only.

## Caveats

- The observation errors are a choice (see above), and the years are treated
  as independent: the posteriors are probably too narrow.
- The prior matters, especially where the observations do not constrain the
  parameters, or where the posterior reaches the prior bounds.
- WGMS seasonal balances are measured either at fixed dates or with the
  stratigraphic method: the model's October-April / May-September split is an
  approximation.
- The geometry is fixed: this calibrates the MB model, not the glacier evolution.